In [1]:
# Python Standard Library
import sys
from os.path import join

# Community Modules
from tqdm import tqdm
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

# My Modules
sys.path.insert(0, "../code")
import measure_signal as ms
import dataset_utils as du
import spectral_sparklines as sp

In [2]:
df_closer_look = pd.read_parquet("../data/forSNR/closer_look.parquet")
df_closer_look.sort_values(by="post_injection_range", ascending=False, inplace=True)
df_closer_look.reset_index(inplace=True, drop=True)
df_closer_look

,SN Name,Spectrum Phase,Spectrum Cardinality,SN Subtype,SN Maintype,SN Subtype ID,SN Maintype ID,SNR,S (SNR),N (SNR),...,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24,post_injection_range
0,sn2004dt,-1.000,1,Ia-norm,Ia,0,0,5795.302733,31.618871,0.005456,...,-0.970480,1.066595,0.854407,2.035191,-3.774301,6.944276,-2.374301,-3.680534,-2.390144,21.988639
1,sn2006x,1.300,1,Ia-norm,Ia,0,0,5544.662228,20.202563,0.003644,...,1.219080,-0.291781,0.390276,-0.975217,-0.953956,-2.609620,3.087096,-3.042977,-0.843497,12.648668
2,sn2002he,-3.200,1,Ia-norm,Ia,0,0,2792.582167,13.024933,0.004664,...,0.970256,2.113966,-0.092081,1.512320,0.750895,-0.595360,0.957644,0.418069,1.121849,7.174206
3,sn2004dt,-3.000,1,Ia-norm,Ia,0,0,767.031086,11.448633,0.014926,...,-0.139531,-0.309572,0.196527,-0.726368,-0.536373,-1.337117,-0.407065,0.140905,-0.851154,7.103497
4,sn2003du,20.700,1,Ia-norm,Ia,0,0,1450.331446,8.881133,0.006124,...,-0.356309,-0.589026,-0.784691,-0.069588,-1.279710,-0.998290,0.601248,0.409570,-0.608912,5.940506
5,sn1994d,12.800,1,Ia-norm,Ia,0,0,1882.360125,9.186518,0.004880,...,1.075877,-0.054358,0.100406,0.531443,1.161328,0.248537,-0.828705,-0.436697,0.416737,5.893248
6,sn2008c,4.400,1,Ia-norm,Ia,0,0,610.631351,7.916669,0.012965,...,0.749990,0.358300,0.461298,0.659872,0.709691,-0.206147,-0.129418,0.195671,-0.306533,5.286865
7,sn2001e,12.200,1,Ia-norm,Ia,0,0,1065.864822,8.621032,0.008088,...,0.852993,0.066407,0.402705,1.489069,1.080033,0.313936,-0.459750,0.645018,0.237442,5.265685
8,sn1998dk,10.200,1,Ia-norm,Ia,0,0,579.369726,7.234791,0.012487,...,0.041522,0.364032,1.532418,-1.151350,0.721774,0.538136,0.226879,0.638155,0.795832,4.804586
9,sn2003cg,-1.000,1,Ia-norm,Ia,0,0,733.574328,8.302043,0.011317,...,0.593054,0.552480,0.658009,0.230484,0.048542,-0.015487,0.528684,-0.468474,1.368531,4.770641


In [3]:
df_dataset = pd.read_parquet("../data/forSNR/deprecated_4/dataset.parquet")
wvl, spectra, df_meta = du.unpack_dataset(df_dataset)
num_spectra, num_wvl = spectra.shape

new_SNR = 10
rng = np.random.default_rng(1415)

new_noise = sp.dataset_make_new_noise(
    new_SNR,
    df_meta,
    num_spectra,
    num_wvl,
    rng)

In [4]:
dataset_inds = []
for i in range(df_closer_look.shape[0]):
    name = df_closer_look.loc[i, "SN Name"]
    phase = df_closer_look.loc[i, "Spectrum Phase"]
    card = df_closer_look.loc[i, "Spectrum Cardinality"]
    mask = df_dataset["SN Name"] == name
    mask &= df_dataset["Spectrum Phase"] == phase
    mask &= df_dataset["Spectrum Cardinality"] == card
    ind = df_dataset[mask].index.to_numpy()
    assert ind.size == 1
    dataset_inds.append(ind[0])

df_dataset = df_dataset.loc[dataset_inds]
df_dataset.reset_index(inplace=True, drop=True)
# df_dataset

In [5]:
new_noise = new_noise[dataset_inds]

In [6]:
wvl, spectra, df_meta = du.unpack_dataset(df_dataset)
num_spectra, num_wvl = spectra.shape

In [7]:
for i in range(df_closer_look.shape[0]):

    # Manual re-calculation of SNR
    specsnr_a = ms.SpectrumSNR(
        df_closer_look.loc[i, "SN Name"],
        df_closer_look.loc[i, "SN Subtype"],
        df_closer_look.loc[i, "Spectrum Phase"],
        wvl,
        df_closer_look.loc[i, wvl.astype(str)].to_numpy())
    specsnr_a.execute_algorithm(df_closer_look.loc[i])

    # Injected Noise to SNR = 10
    specsnr_b = ms.SpectrumSNR(
        df_dataset.loc[i, "SN Name"],
        df_dataset.loc[i, "SN Subtype"],
        df_dataset.loc[i, "Spectrum Phase"],
        wvl,
        df_dataset.loc[i, wvl.astype(str)].to_numpy())
    specsnr_b.execute_algorithm(df_dataset.loc[i], new_noise=new_noise[i])

    # Original spectrum
    specsnr_c = ms.SpectrumSNR(
        df_dataset.loc[i, "SN Name"],
        df_dataset.loc[i, "SN Subtype"],
        df_dataset.loc[i, "Spectrum Phase"],
        wvl,
        df_dataset.loc[i, wvl.astype(str)].to_numpy())
    specsnr_c.execute_algorithm(df_dataset.loc[i])

    fig, axes = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 6))
    fig.suptitle(f'{df_closer_look.loc[i, "SN Subtype"]} | {df_closer_look.loc[i, "SN Name"]} at {df_closer_look.loc[i, "Spectrum Phase"]} ({df_closer_look.loc[i, "Spectrum Cardinality"]})')
    
    axes[0].set_xlim((4500, 7000))


    axes[0].set_title("Original Spectrum")
    axes[0].plot(wvl, specsnr_c.spectrum)
    y_mid = np.sum(axes[0].get_ylim()) / 2
    x_lo, x_hi = axes[0].get_xlim()
    spec_max = (specsnr_c.signal + specsnr_c.noise).max()
    spec_min = (specsnr_c.signal + specsnr_c.noise).min()
    text_info = (
        f"$SNR = {specsnr_c.SNR:.2f}$" "\n"
        f"$S = {specsnr_c.S:.2e}$" "\n"
        f"$N = {specsnr_c.N:.2e}$" "\n"
        f"$\sigma = {specsnr_c.denoising_parameter}$" "\n"
        f"Range = {spec_max-spec_min:.2f}" #"\n"
    )
    axes[0].text(x_hi*1.025, y_mid, text_info, ha="left", va="center")
    

    axes[1].set_title("Injected Noise to SNR = 10")
    axes[1].plot(wvl, specsnr_b.signal + specsnr_b.noise)
    y_mid = np.sum(axes[1].get_ylim()) / 2
    x_lo, x_hi = axes[1].get_xlim()
    spec_max = (specsnr_b.signal + specsnr_b.noise).max()
    spec_min = (specsnr_b.signal + specsnr_b.noise).min()
    text_info = (
        f"$SNR = {specsnr_b.SNR:.2f}$" "\n"
        f"$S = {specsnr_b.S:.2e}$" "\n"
        f"$N = {specsnr_b.N:.2e}$" "\n"
        f"$\sigma = {specsnr_b.denoising_parameter}$" "\n"
        f"Range = {spec_max-spec_min:.2f}" #"\n"
    )
    axes[1].text(x_hi*1.025, y_mid, text_info, ha="left", va="center")

    axes[2].set_title("Manual re-calculation of SNR")
    axes[2].plot(wvl, specsnr_a.signal)
    y_mid = np.sum(axes[2].get_ylim()) / 2
    x_lo, x_hi = axes[2].get_xlim()
    spec_max = (specsnr_a.signal + specsnr_a.noise).max()
    spec_min = (specsnr_a.signal + specsnr_a.noise).min()
    text_info = (
        f"$SNR = {specsnr_a.SNR:.2f}$" "\n"
        f"$S = {specsnr_a.S:.2e}$" "\n"
        f"$N = {specsnr_a.N:.2e}$" "\n"
        f"$\sigma = {specsnr_a.denoising_parameter}$" "\n"
        f"Range = {spec_max-spec_min:.2f}" #"\n"
    )
    axes[2].text(x_hi*1.025, y_mid, text_info, ha="left", va="center")

    axes[0].plot(specsnr_c.pc_wvl, specsnr_c.pseudo_cont, c="tab:green")
    axes[1].plot(specsnr_b.pc_wvl, specsnr_b.pseudo_cont, c="tab:green")
    axes[2].plot(specsnr_a.pc_wvl, specsnr_a.pseudo_cont, c="tab:green")
    

    if specsnr_a.useBlu:
        axes[0].axvspan(
            specsnr_c.wvl[specsnr_c.blu_inds][-1],
            specsnr_c.wvl[specsnr_c.blu_inds][0],
            color="tab:blue",
            alpha=0.25)
        axes[1].axvspan(
            specsnr_b.wvl[specsnr_b.blu_inds][-1],
            specsnr_b.wvl[specsnr_b.blu_inds][0],
            color="tab:blue",
            alpha=0.25)
        axes[2].axvspan(
            specsnr_a.wvl[specsnr_a.blu_inds][-1],
            specsnr_a.wvl[specsnr_a.blu_inds][0],
            color="tab:blue",
            alpha=0.25)

    if specsnr_a.useRed:
        axes[0].axvspan(
            specsnr_c.wvl[specsnr_c.red_inds][-1],
            specsnr_c.wvl[specsnr_c.red_inds][0],
            color="tab:red",
            alpha=0.25)
        axes[1].axvspan(
            specsnr_b.wvl[specsnr_b.red_inds][-1],
            specsnr_b.wvl[specsnr_b.red_inds][0],
            color="tab:red",
            alpha=0.25)
        axes[2].axvspan(
            specsnr_a.wvl[specsnr_a.red_inds][-1],
            specsnr_a.wvl[specsnr_a.red_inds][0],
            color="tab:red",
            alpha=0.25)    

    plt.tight_layout()
    # display(plt.gcf())
    plt.savefig(f"../figures/closer_look/{i:0>2}.png")
    plt.close()
    # break